FRAME TO FRAME INFERENCE & EVALUATION

In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
import pickle
import warnings
import gzip
import scipy.io
from scipy import signal
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

# 8k Net trained
# workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
workspace_dir = '/home/adelval/BTS/TFM/afterburner8k_win20/'
# 16k Net trained
# workspace_dir = '/home/adelval/BTS/TFM/test/'

reduced_net = False

sys.path.append(workspace_dir + 'src/net1')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [2]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

PREEMPHASIS OPTIMIZATION

In [3]:
import numpy as np

def original_preemphasis(x):
    x = list(x) #make sure it is a list
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))):
        x[i] = x[i] - x[i-1] * 0.97 if i > 0 else x[i]
    x[0] = x0
    return np.array(x)

def optimized_preemphasis(x, alpha=0.97):
    x = np.array(x, dtype=np.float64)
    x[1:] = x[1:] - alpha * x[:-1]
    x[0] = x[0] * (1 - alpha)
    return x

# Example input
input_signal = [1.0, 2.0, 3.0, 4.0, 5.0]

# Calculate outputs
original_output = original_preemphasis(input_signal)
optimized_output = optimized_preemphasis(input_signal)

# Print outputs
print("Original Output:", original_output)
print("Optimized Output:", optimized_output)

# Check for equality
print("Outputs are equal:", np.allclose(original_output, optimized_output))

#Example with more values
input_signal2 = np.linspace(0,10,1000)
start = time.time()
original_output2 = original_preemphasis(input_signal2)
print("Original time:", time.time()-start)
start = time.time()
optimized_output2 = optimized_preemphasis(input_signal2)
print("Optimized time:", time.time()-start)

print("Outputs are equal for larger array:", np.allclose(original_output2, optimized_output2))

Original Output: [0.03 1.03 1.06 1.09 1.12]
Optimized Output: [0.03 1.03 1.06 1.09 1.12]
Outputs are equal: True
Original time: 0.0009541511535644531
Optimized time: 7.653236389160156e-05
Outputs are equal for larger array: True


In [41]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

def optimized_preemphasis(x, alpha=0.97):
    x = np.array(x, dtype=np.float64)
    x[1:] = x[1:] - alpha * x[:-1]
    x[0] = x[0] * (1 - alpha)
    return x

# Divide x into overlapping frames of fixed length without extending to slide last frame
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010, Mw=20):
    # Mw limit the number of windows
    N = int(Ns * fs)            # Number of samples in each window 640
    M = int(Ms * fs)            # Step size (number of samples between window starts) 160
    n = (len(x) + M - 1) // M   # Number of frames 23
    # print("Number of frames", n)    
    # T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    # Se ignora el padding porque la ventana se deslizará
    # if T > len(x):
    #     print("rellena con ceros")
    #     xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, Mw*M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    xa = xa[ind.astype(int).T].astype(np.float32)
    # print(f'Window frames {xa[19,:5]}')

    return xa

def hamming(X):
    w = np.hamming(X.shape[1])
    print(f'Hamming window shape {w.shape}')
    return X * w

def fft(X, NFFT):
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=1)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:, :(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals



def frame_fft(data, fs, w, nfft, max_windows, hamming_win):
    

    pre_offset = time.time()
    x = offset(data)
    global accum_offset 
    accum_offset += (time.time()-pre_offset)

    # Emphasis to increase the amplitude of high freq
    pre_emphasis = time.time()
    x = optimized_preemphasis(x)
    # print(f"Preemphasis time: {(time.time()-pre_emphasis)*1000} ms")
    global accum_preemphasis 
    accum_preemphasis += (time.time()-pre_emphasis)

    X = windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows) * hamming_win

    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])  # Instead of calling a funtion, do directly

    # print(f"la shape de Xfft es {Xfft.shape}")
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    
    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    # print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [36]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb



def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


def frame_fb_mfcc(data, fs, B, w, nfft, max_windows, fb, dct, hamming_win):
    x = offset(data)
    x = optimized_preemphasis(x)
    
    X = windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_windows) * hamming_win

    Xfft = fft(X, nfft[0])
    Xb = np.log(Xfft.dot( fb[0] ) + 1)
    Xc = Xb.dot(dct[0])                          
    
    X = np.concatenate( [Xb, Xc], 1 )
    
    X = np.asarray(X, dtype=np.float32)
    
    return X

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc, mu, std):
    # Normalization of fbmfcc
    frame_fbmfcc -= mu
    frame_fbmfcc /= std + 1e-6

    return frame_fbmfcc
    

LOADS FOR FULL NET

In [ ]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

if (reduced_net):
    input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions_rn.pkl') 
else:
    input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 


print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append( workspace_dir + 'src/net')


if (reduced_net):
    from net_snr import Net_snr
else:
    from net_snr_original import Net_snr



net_snr = Net_snr(input_dim, output_dim, cuda=True, single_gpu=True)

if (reduced_net):
    net_snr.load_theta( workspace_dir + 'data/model/theta_last_rn')
else:
    net_snr.load_theta( workspace_dir + 'data/model/theta_last')



LOADS FOR WINDOWS NET

In [6]:
# Load model dimensions and weights for windowing

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)


input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 


print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))

# Load net for windowing
sys.path.append( workspace_dir + 'src/net1')
from net_snr import Net_snr 
net_snr = Net_snr(input_dim, output_dim, cuda=True, single_gpu=True)
net_snr.load( workspace_dir + 'data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:
    nb_params: 29.99M
    cuda: True
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    restoring epoch: 500
    restoring opt: adama, lr: 0.000019
    opt: adama, 1.94812e-05, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 1.9481214405699424e-05
    weight_decay: 0
)
    reading /home/adelval/BTS/TFM/afterburner8k_win20/data/model/theta_last


500

In [7]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    n_frames, fft_fb = x.shape
    x = x.reshape(1, n_frames, fft_fb)
    #x = x.unsqueeze(0) # Batch dimension
    # print(f'Las dimensiones de la muestra a evaluar son: {x.shape}')

    snr = net_snr.predict(x)
    snr = to_numpy(snr.squeeze())
    # print(f'La máscara del frame caculado es de {snr.shape}')
    
    #scipy.io.savemat(f, mdict={'snr': snr})
    # x = to_numpy(x.squeeze())
    # snr = to_numpy(snr.squeeze())
    return snr


In [8]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    # print(f'el tipo de yw es {yw.dtype}')
    # print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    #it = int(np.floor((data.size-frame)/shift))
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        # print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        # print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        # print(Xfft.shape)
        # print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        # print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        # print(f'El tamaño de la salida del filtro sera {outf.shape}')
        # print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        # print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        # print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # print(f'El tipo de la señal es {data.dtype}')
    # print(f'El tipo de la red es {snr_net.dtype}')
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    # print(f'El tipo de la señal es {data.dtype} con ruido añadido')
    snr_net = snr_net.reshape(-1,1) # para darle 2-D
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    # print(f'El filtro será {filt.dtype}')
    # print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


In [42]:
# Parámetros
fs=8000
B=[32]
w=[0.040]
m=0.01
nfft=[1024]
gmin = 0.0562
min_windows = 4
max_windows = 20
diezmation_factor = 2

# Cálculo de los filtros
N =[int(wi * fs) for wi in w]
F = [int(nffti/2) for nffti in nfft]
fb_time = time.time()
fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
print(f'El tiempo de cálculo de los filtros es {(time.time() - fb_time)*1000} ms')
dct_time = time.time()
dct = [ f_base_dct(Bi) for Bi in B] 
print(f'El tiempo de cálculo de las bases dct es {(time.time() - dct_time)*1000} ms')

# Media y desviación para la normalización
file = workspace_dir + 'data/model/fe1_norm1.pkl'
mu, std = read_pkl(file)


# Ventana de hamming
hamming_win = np.hamming(fs * w[0])

# Variables globales para el cálculo de los tiempos
accum_offset = 0
accum_preemphasis = 0
accum_windowing = 0
accum_fft = 0
accum_log = 0
accum_fb_mfcc = 0
accum_norm = 0 
accum_inf = 0


## SELECCIÓN DE AUDIO
# AUDIOS 16K
# x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/livekit/audio_received.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner_2023/afterburner16k/data/audio/callcenter_movistar/AUDIOS/Audio-AudioModule_708351_AudioChannel_22403522_19-May-2023_15.53.49.097.wav']

# AUDIOS 8K
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/11-CH0_C01_construction_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/36635c2b-ca8a-5d7c-abb6-42981f7613e9.wav']

print(f'El audio elegido es {x_test[0]}')

if(reduced_net):
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16.wav')
    print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
else:
    output_enh_file = os.path.basename(x_test[0])
    output_enh = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_float64.wav')
    output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                +'w_'+ str(int(w[0]*1000)) +'ms_s'+ str(int(m*1000)) +'_int16.wav')
    # print(f'El audio mejorado es {output_enh} de tipo float64 y\n {output_enh_int} de tipo int16')
    print(f'El audio mejorado es {output_enh_int} de tipo int16')


audio, fs = read_audio(x_test[0])
print(f'La frecuencia de muestreo es {fs} Hz')
print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

# #------------------------SNR PRE-----------------------------#
# pre_vad = compute_vad(audio, fs, w[0], m, nfft[0])
# snr_prev = int(wada_snr(audio, fs, pre_vad))
# print('snr(wada)=%idB, file: %s' % (snr_prev, x_test[0]))
# #------------------------------------------------------------#

diff_acum = 0
diff_inf_acum = 0


frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
print(f'Tamaño de frame: {frame_samples} samples')
shift_size = m  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
print(f'Desplazamiento: {shift_samples} samples')
window_size = w[0]  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640
print(f'Tamaño de ventana: {window_samples} samples')
window_inference_min = int((min_windows + (window_size/frame_size)-1) * frame_samples)
window_inference_max = int((max_windows + (window_size/frame_size)-1) * frame_samples)
print(f'El buffer retendra desde {window_inference_min} hasta {window_inference_max} samples')
buffer_frame = np.zeros(0)  # Buffer de ventana recibida

it = 0
snr_frame_mask = np.ones((512,min_windows)) # Inicializado con la duración de la ventana de inferencia
yenh = np.zeros(len(audio)) # Inicializado con la duración del audio original

# CALCULO DE LA MÁSCARA SNR 
for n_frame in range(int((len(audio)/fs)*100)):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]
    print(f'\n{n_frame} frame de {len(frame)} --> {frame[:10]}')
    # Concatenar el frame recibido al buffer `buffer_frame`
    buffer_frame = np.concatenate([buffer_frame, frame])
    start_time = time.time()

    if len(buffer_frame) >= window_samples:
        # Extraer las primeras 640 muestras como ventana completa
        if len(buffer_frame) < window_inference_max:
            work_window = buffer_frame[it*shift_samples:it*shift_samples+window_samples]
            # print("Iteración número:", n_frame)
            # print("Ventana acumulada", work_window[:10])
            # print("Ventana acumulada", work_window[160:170])
            # print("Ventana acumulada", work_window[320:330])
            # print("Ventana acumulada", work_window[480:490])
            it += 1 # Number of windows received
            if len(buffer_frame) >= window_inference_min:
                # print("Reached MIN WINDOW --> STRATING INFERENCE")
                work_inf_frames = buffer_frame[:window_inference_max]
                # print(f'0 frame  {work_inf_frames[:10]}')
                # print(f'1 frame  {work_inf_frames[frame_samples:frame_samples+10]}')
                # print(f'2 frame  {work_inf_frames[2*frame_samples:2*frame_samples+10]}')
                # print(f'3 frame  {work_inf_frames[3*frame_samples:3*frame_samples+10]}')
                start_fft = time.time()
                fft_windows = frame_fft(work_inf_frames, fs, w, nfft, it, hamming_win)
                accum_fft += (time.time()-start_fft)
                # print(f'Tiempo de procesamiento de la FFT del frame {n_frame} es de {(time.time()-start_fft)*1000} ms')
                # print("0 Vector 2D con FFT para cada frame por row \n",  fft_windows[0,:10])
                # print("1 Vector 2D con FFT para cada frame por row \n",  fft_windows[1,:10])
                # print("2 Vector 2D con FFT para cada frame por row \n",  fft_windows[2,:10])
                # print("3 Vector 2D con FFT para cada frame por row \n",  fft_windows[3,:10])
                start_log = time.time()
                fft_windows_log = log_scale(fft_windows)
                # print(f'Tiempo de procesamiento de la escala log del frame {n_frame} es de {(time.time()-start_log)*1000} ms')
                accum_log += (time.time()-start_log)
                # print(" 0 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[0,:10])
                # print(" 1 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[1,:10])
                # print(" 2 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[2,:10])
                # print(" 3 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[3,:10])
                start_fb = time.time()
                fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w, nfft, it, fb, dct, hamming_win)
                accum_fb_mfcc += (time.time()-start_fb)
                # print(f'Tiempo de procesamiento de la FB MFCC del frame {n_frame} es de {(time.time()-start_fb)*1000} ms')
                # print(f'0 Filtro Mel {fb_windows[0,:5]}')
                # print(f'1 Filtro Mel {fb_windows[1,:5]}')
                # print(f'2 Filtro Mel {fb_windows[2,:5]}')
                # print(f'3 Filtro Mel {fb_windows[3,:5]}')
                start_norm = time.time()
                fb_windows_norm = norm_fb_frame(fb_windows, mu, std)
                accum_norm += (time.time()-start_norm)
                # print(f'Tiempo de procesamiento de la normalización del frame {n_frame} es de {(time.time()-start_norm)*1000} ms')
                # print(f'0 Filtro Mel normalizado {fb_windows_norm[0,:5]}')
                # print(f'1 Filtro Mel normalizado {fb_windows_norm[1,:5]}')
                # print(f'2 Filtro Mel normalizado {fb_windows_norm[2,:5]}')
                # print(f'3 Filtro Mel normalizado {fb_windows_norm[3,:5]}')
                windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )
                # print("El tamaño de x08k tras la concatenacion es: ",windows_concat.shape)
                # print(f'0 La concatenacion resulta  {windows_concat[0,:5]}  y {windows_concat[0,512:517]}')
                # print(f'1 La concatenacion resulta  {windows_concat[1,:5]}  y {windows_concat[1,512:517]}')
                # print(f'2 La concatenacion resulta  {windows_concat[2,:5]}  y {windows_concat[2,512:517]}')
                # print(f'3 La concatenacion resulta  {windows_concat[3,:5]}  y {windows_concat[3,512:517]}')

                start_prof = time.time()
                snr_frame_mask = net_eval(windows_concat)
                accum_inf += (time.time()-start_prof)
                # print(f'Tiempo de procesamiento de la inferencia del frame {n_frame} es de {(time.time()-start_prof)*1000} ms')

                
                # print(snr_frame_mask[0,:10])
                # print(snr_frame_mask[1,:10])
                # print(snr_frame_mask[2,:10])
                # print(snr_frame_mask[3,:10])
                snr_frame_mask = snr_frame_mask.T
                # print(f'La máscara {it} calculada es de {snr_frame_mask.shape}')
                # print(f'Frames restantes en el buffer {len(buffer_frame)/frame_samples}')

        # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
        else:
            work_window = buffer_frame[window_inference_max-window_samples:] # Los últimos frames del buffer
            # print("Iteración número:", n_frame)
            # print("Ventana acumulada", work_window[:10])
            # print("Ventana acumulada", work_window[160:170])
            # print("Ventana acumulada", work_window[320:330])
            # print("Ventana acumulada", work_window[480:490])
            # print("Reached MAX WINDOW --> Starting Inference")
            work_inf_frames = buffer_frame[:window_inference_max]
            # print(f'0 frame  {work_inf_frames[:10]}')
            # print(f'1 frame  {work_inf_frames[frame_samples:frame_samples+10]}')
            # print(f'2 frame  {work_inf_frames[2*frame_samples:2*frame_samples+10]}')
            # print(f'3 frame  {work_inf_frames[3*frame_samples:3*frame_samples+10]}')
            # print(f'23 frame {work_inf_frames[22*frame_samples:22*frame_samples+10]}')
            start_fft = time.time()
            fft_windows = frame_fft(work_inf_frames, fs, w, nfft, max_windows, hamming_win)
            accum_fft += (time.time()-start_fft)
            # print(f'Tiempo de procesamiento de la FFT del frame {n_frame} es de {(time.time()-start_fft)*1000} ms')
            # print("0 Vector 2D con FFT para cada frame por row \n",  fft_windows[0,:10])
            # print("1 Vector 2D con FFT para cada frame por row \n",  fft_windows[1,:10])
            # print("2 Vector 2D con FFT para cada frame por row \n",  fft_windows[2,:10])
            # print("3 Vector 2D con FFT para cada frame por row \n",  fft_windows[3,:10])
            # print("20 Vector 2D con FFT para cada frame por row \n", fft_windows[19,:10])
            start_log = time.time()
            fft_windows_log = log_scale(fft_windows)
            accum_log += (time.time()-start_log)
            # print(f'Tiempo de procesamiento de la escala log del frame {n_frame} es de {(time.time()-start_log)*1000} ms')
            # print(" 0 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[0,:10])
            # print(" 1 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[1,:10])
            # print(" 2 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[2,:10])
            # print(" 3 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[3,:10])
            # print("20 Vector 2D con FFT en escala log para cada frame por row \n", fft_windows_log[19,:10])
            start_fb = time.time()
            fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w, nfft, max_windows, fb, dct, hamming_win)
            accum_fb_mfcc += (time.time()-start_fb)
            # print(f'Tiempo de procesamiento de la FB MFCC del frame {n_frame} es de {(time.time()-start_fb)*1000} ms')
            # print(fb_windows.shape)
            # print(f'0 Filtro Mel {fb_windows[0,:5]}')
            # print(f'1 Filtro Mel {fb_windows[1,:5]}')
            # print(f'2 Filtro Mel {fb_windows[2,:5]}')
            # print(f'3 Filtro Mel {fb_windows[3,:5]}')
            # print(f'19 Filtro Mel {fb_windows[19,:5]}')
            start_norm = time.time()
            fb_windows_norm = norm_fb_frame(fb_windows, mu, std)
            accum_norm += (time.time()-start_norm)
            # print(f'Tiempo de procesamiento de la normalización del frame {n_frame} es de {(time.time()-start_norm)*1000} ms')
            # print(f'0 Filtro Mel normalizado {fb_windows_norm[0,:5]}')
            # print(f'1 Filtro Mel normalizado {fb_windows_norm[1,:5]}')
            # print(f'2 Filtro Mel normalizado {fb_windows_norm[2,:5]}')
            # print(f'3 Filtro Mel normalizado {fb_windows_norm[3,:5]}')
            # print(f'19 Filtro Melnormalizado {fb_windows_norm[19,:5]}')
            # start_concat = time.time()
            windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )
            # print(f'Tiempo de procesamiento de la concatenación del frame {n_frame} es de {(time.time()-start_concat)*1000} ms')
            # print("El tamaño de x08k tras la concatenacion es: ",windows_concat.shape)
            # print(f'0 La concatenacion resulta  {windows_concat[0,:5]}  y {windows_concat[0,512:517]}')
            # print(f'1 La concatenacion resulta  {windows_concat[1,:5]}  y {windows_concat[1,512:517]}')
            # print(f'2 La concatenacion resulta  {windows_concat[2,:5]}  y {windows_concat[2,512:517]}')
            # print(f'3 La concatenacion resulta  {windows_concat[3,:5]}  y {windows_concat[3,512:517]}')
            # print(f'19 La concatenacion resulta {windows_concat[19,:5]} y {windows_concat[19,512:517]}')

            start_prof = time.time()
            # Process inference with factor 2 diezmation
            if(n_frame % diezmation_factor == 0):
                snr_frame_mask = net_eval(windows_concat)
                snr_frame_mask = snr_frame_mask.T
            accum_inf += (time.time()-start_prof)
            # print(f'Tiempo de procesamiento de la inferencia del frame {n_frame} es de {(time.time()-start_prof)*1000} ms')
            # print(snr_frame_mask.shape)
            # print(snr_frame_mask[0,:10])
            # print(snr_frame_mask[1,:10])
            # print(snr_frame_mask[2,:10])
            # print(snr_frame_mask[3,:10])
            # print(snr_frame_mask[19,:10])
            #Desplazar las muestras en `buffer_frame` para la próxima ventana
            buffer_frame = buffer_frame[shift_samples:]
            # print(f'Frames restantes en el buffer {len(buffer_frame)/frame_samples}')

        # Aqui haría la evaluacion con la máscara pertinente (para las primeras 3 ventanas sin máscara calculada)
        # cnt = int(n_frame - w[0]/m) Ajustar al tamaño de la ventana
        cnt = n_frame - 3
        # print(f'EVALUATION OF WINDOW {cnt}')
        x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
        # print(f'El frame sin enventanado resulta {x[:10]}')
        # print(f'Se le aplica la mascara {snr_frame_mask[:10,-1]}')
        xenh, filt = noiseReduction(x, snr_frame_mask[:,-1], fs, window_samples, shift_samples, nfft[0], gmin)
        # print(f'xenh es de type {xenh.dtype}')
        # print(f'Las dimensiones del filtro son {filt.shape}')
        # print(f'Window processed {cnt} {xenh.shape} {yenh[cnt*shift_samples : cnt*shift_samples+window_samples].shape}')
        slice_size = min(len(yenh) - cnt * shift_samples, window_samples)
        # print(f'El slice_size es {slice_size}')
        yenh[cnt * shift_samples : cnt * shift_samples + slice_size] += xenh[0:slice_size]
        # print(f'El audio previamente mejorado resulta   {xenh[cnt*shift_samples : cnt*shift_samples+10]}')
        # print(f'El audio mejorado resulta                                   {yenh[cnt*shift_samples : cnt*shift_samples+10]}')
        # print(f'El audio mejorado resulta float16                           {yenh[cnt*shift_samples : cnt*shift_samples+10].astype(np.float16)}')
        # print(f'El audio mejorado resulta {yenh.dtype}                      {(yenh[cnt*shift_samples : cnt*shift_samples+10]*2**15)}')
        # print(f'El audio mejorado resulta {yenh.astype(np.int16).dtype}     {(yenh[cnt*shift_samples : cnt*shift_samples+10]*2**15).astype(np.int16)}')
        # diff_inf_acum += end_prof - start_prof

    end_time = time.time()
    diff_acum += end_time - start_time
    print(f'Tiempo de procesamiento del frame {n_frame} completo es de {(end_time-start_time)*1000} ms')

print(f'\n|-----------------------------FINAL TIME STATS-------------------------------|')
print(f'  El tiempo medio de offset es de {(accum_offset/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de preemphasis es de {(accum_preemphasis/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la FFT es de {(accum_fft/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la escala log es de {(accum_log/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la FB MFCC es de {(accum_fb_mfcc/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la normalización es de {(accum_norm/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento de la inferencia es de {(accum_inf/(n_frame-6))*1000} ms')
print(f'  El tiempo medio de procesamiento total es de {(diff_acum/n_frame)*1000} ms')
print(f'|----------------------------------------------------------------------------|\n')

print(f'El audio tiene len {len(yenh)} y tipo {yenh.dtype}')
print(f'El audio mejorado resulta {yenh[:10]}')

yenh = yenh/3     # 4 because in OverLapAdd we sum 4 times the frame

yenh = np.array(yenh*(2 ** 15), dtype=np.int16)     # set int16 wav format

yenh_clipped = np.clip(yenh*2**15, -32768, 32767).astype(np.int16)
wavfile.write(output_enh_int,fs,yenh)


#--------------------------SNR POST--------------------------#
# post_vad = compute_vad(yenh_clipped, fs, w[0], m, nfft[0])
# snr_post = int(wada_snr(yenh_clipped, fs, pre_vad))
# print('snr(wada)=%idB, file: %s' % (snr_post, output_enh))
#------------------------------------------------------------#




El tiempo de cálculo de los filtros es 1.7924308776855469 ms
El tiempo de cálculo de las bases dct es 1.8420219421386719 ms
El audio elegido es /home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav
El audio mejorado es /home/adelval/BTS/TFM/audios/enh/6-CH0_C01_traffic_15dB_f2f_enh_HM_2_8k_4to20w_40ms_s10_int16.wav de tipo int16
La frecuencia de muestreo es 8000 Hz
La duración del audio es 3.168125 segundos y 25345 muestras
Tamaño de frame: 80 samples
Desplazamiento: 80 samples
Tamaño de ventana: 320 samples
El buffer retendra desde 560 hasta 1840 samples

0 frame de 80 --> [ 206  968  782  781 1234  990  747  592  633  -31]
Tiempo de procesamiento del frame 0 completo es de 0.0011920928955078125 ms

1 frame de 80 --> [-462  109  153  187  159  245  631  313  386  773]
Tiempo de procesamiento del frame 1 completo es de 0.0007152557373046875 ms

2 frame de 80 --> [ 608  209  414  321  255  165    3  -22 -555  -20]
Tiempo de procesamiento del frame 2 compl

In [ ]:
sys.path.append('/home/adelval/BTS/TFM/quality')
from srmr import *

x_srmr = audio

try:
    srmr_value_before, _ = srmr(x_srmr, fs, n_cochlear_filters=23, low_freq=125, min_cf=4, max_cf=128, fast=True, norm=False)
    srmr_value_before = np.round(srmr_value_before,2)
except Exception as e:
    srmr_value_before = np.nan
    print(f'Error in SRMR calculation: {e}')

y_srmr = yenh_clipped

try:
    srmr_value_after, _ = srmr(y_srmr, fs, n_cochlear_filters=23, low_freq=125, min_cf=4, max_cf=128, fast=True, norm=False)
    srmr_value_after = np.round(srmr_value_after,2)
except Exception as e:
    srmr_value_after = np.nan
    print(f'Error in SRMR calculation: {e}')


print(f'SRMR: {srmr_value_before} before enhancement')
print(f'SRMR: {srmr_value_after} after enhancement')